In [22]:
import pandas as pd 
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
print('imported')

imported


In [23]:
# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

In [24]:
dirs = '/home/ab303el/Documents/virtual internship/CodeAlpha Internship/data/raw/'
data = 'climate_data.csv'
data = dirs + data

In [25]:
df = pd.read_csv(data)

df.head(2)

,Country,Disaster_Type,Date,Original_Title
0,Indonesia,Earthquake,Aug 2026,Indonesia: Earthquake - Aug 2026
1,Colombia,Earthquake,Aug 2026,Colombia: Earthquake - Aug 2026


In [26]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11799 entries, 0 to 11798
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Country         11799 non-null  str  
 1   Disaster_Type   11799 non-null  str  
 2   Date            11799 non-null  str  
 3   Original_Title  11799 non-null  str  
dtypes: str(4)
memory usage: 368.8 KB


In [27]:
def audit(df):
    return pd.DataFrame(
        {
            'dtype':df.dtypes,
            'nulls':df.isnull().sum(),
            'null_%':(df.isnull().mean() * 100).round(2),
            'unique':df.nunique(),
            'unique_%':(df.nunique() / len(df) * 100).round(2),
            'zeros':(df==0).sum(),
            'sample':df.iloc[0]
        }
    )
audit(df)

,dtype,nulls,null_%,unique,unique_%,zeros,sample
Country,str,0,0.00,260,2.20,0,Indonesia
Disaster_Type,str,0,0.00,431,3.65,0,Earthquake
Date,str,0,0.00,375,3.18,0,Aug 2026
Original_Title,str,0,0.00,2728,23.12,0,Indonesia: Earthquake - Aug 2026


set data types correctly

In [28]:
# regular expression
date_pattern = r'^[A-Za-z]{3}\s+\d{4}$'

# we Remove 'Date' columns with corrupt data by comparing with our date pattern 
corrupt_rows = df[~df['Date'].astype(str).str.contains(date_pattern, na=False)]

print(corrupt_rows)


       Country    Disaster_Type  Date                        Original_Title
55     Pacific  Dengue Outbreak  2025  Pacific: Dengue Outbreak - 2025-2026
133    Pacific  Dengue Outbreak  2025  Pacific: Dengue Outbreak - 2025-2026
214    Pacific  Dengue Outbreak  2025  Pacific: Dengue Outbreak - 2025-2026
291    Pacific  Dengue Outbreak  2025  Pacific: Dengue Outbreak - 2025-2026
371    Pacific  Dengue Outbreak  2025  Pacific: Dengue Outbreak - 2025-2026
...        ...              ...   ...                                   ...
11454  Pacific  Dengue Outbreak  2025  Pacific: Dengue Outbreak - 2025-2026
11535  Pacific  Dengue Outbreak  2025  Pacific: Dengue Outbreak - 2025-2026
11615  Pacific  Dengue Outbreak  2025  Pacific: Dengue Outbreak - 2025-2026
11694  Pacific  Dengue Outbreak  2025  Pacific: Dengue Outbreak - 2025-2026
11774  Pacific  Dengue Outbreak  2025  Pacific: Dengue Outbreak - 2025-2026

[209 rows x 4 columns]


we have identified over 209 rows with missaligned datas so we remove those
209 columns as the data is too big to be affected by this rows

In [29]:
# Drop rows directly using the index of our corrupt_rows 
df = df.drop(index=corrupt_rows.index)

# Verify that no corrupt rows remain in Date column
print(df[~df['Date'].astype(str).str.contains(date_pattern, na=False)])


Empty DataFrame
Columns: [Country, Disaster_Type, Date, Original_Title]
Index: []


convert the data type of out column Date from str to datetime

In [30]:
df['Date'] = df['Date'].apply(pd.to_datetime, format='mixed')
df.dtypes

Country                      str
Disaster_Type                str
Date              datetime64[us]
Original_Title               str
dtype: object

In [31]:
df.info()

<class 'pandas.DataFrame'>
Index: 11590 entries, 0 to 11798
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Country         11590 non-null  str           
 1   Disaster_Type   11590 non-null  str           
 2   Date            11590 non-null  datetime64[us]
 3   Original_Title  11590 non-null  str           
dtypes: datetime64[us](1), str(3)
memory usage: 452.7 KB


In [32]:
def audit(df):
    return pd.DataFrame(
        {
            'dtype':df.dtypes,
            'nulls':df.isnull().sum(),
            'null_%':(df.isnull().mean() * 100).round(2),
            'unique':df.nunique(),
            'unique_%':(df.nunique() / len(df) * 100).round(2),
            'zeros':(df==0).sum(),
            'sample':df.iloc[0]
        }
    )
audit(df)

,dtype,nulls,null_%,unique,unique_%,zeros,sample
Country,str,0,0.00,256,2.21,0,Indonesia
Disaster_Type,str,0,0.00,422,3.64,0,Earthquake
Date,datetime64[us],0,0.00,345,2.98,0,2026-08-01 00:00:00
Original_Title,str,0,0.00,2668,23.02,0,Indonesia: Earthquake - Aug 2026


create new column for Continent based on the column Country

In [33]:
Countries = df['Country'].unique()
print(sorted(Countries))

['Afghanistan', 'Afghanistan/Pakistan', 'Afghanistan/Tajikistan', 'Albania', 'Algeria', 'Angola', 'Angola/DR Congo', 'Argentina', 'Argentina/Paraguay', 'Argentina/Uruguay', 'Armenia', 'Asia', 'Australia', 'Azerbaijan', 'Bahamas', 'Balkans', 'Bangladesh', 'Bangladesh and India', 'Bangladesh/Myanmar', 'Bay of Bengal (India & Bangladesh)', 'Belarus', 'Belarus/Russian Fed./Ukraine and Moldova', 'Belize', 'Benin', 'Benin/Nigeria/Togo', 'Bhutan', 'Bolivia', 'Bosnia and Herzegovina', 'Botswana', 'Brazil', 'Brazil/Paraguay', 'Bulgaria', 'Burkina Faso', 'Burundi', 'Burundi/Tanzania', 'Cabo Verde', 'Cambodia', 'Cameroon', 'Cameroon/Chad', 'Cameroon/Equatorial Guinea', 'Cape Verde', 'Caribbean', 'Central African Republic', 'Central America', 'Central America and the Caribbean', 'Central Europe', 'Central and Eastern Europe', 'Central and Latin America', 'Central/Eastern Africa', 'Central/Eastern Europe', 'Central/Southern Europe', 'Chad', 'Chile', 'China', 'Colombia', 'Comoros', 'Congo', 'Congo R

 mapped dictionary of our unique countries


In [34]:
continent_groups = {
    'Africa': (
        'Algeria', 'Angola', 'Angola/DR Congo', 'Benin', 'Benin/Nigeria/Togo', 'Botswana', 
        'Burkina Faso', 'Burundi', 'Burundi/Tanzania', 'Cabo Verde', 'Cameroon', 'Cameroon/Chad', 
        'Cameroon/Equatorial Guinea', 'Cape Verde', 'Central African Republic', 'Central/Eastern Africa', 
        'Chad', 'Comoros', 'Congo', 'Congo River', "Cote d'Ivoire", "Côte d'Ivoire", 'DR Congo', 
        'Djibouti', 'East Africa', 'Egypt', 'Equatorial Guinea', 'Eswatini', 'Ethiopia', 'Gabon', 
        'Gambia', 'Ghana', 'Great Lakes', 'Guinea', 'Guinea-Bissau', 'Horn of Africa', 'Kenya', 
        'Lesotho', 'Liberia', 'Libya', 'Madagascar', 'Malawi', 'Mali', 'Mauritania', 'Mauritius', 
        'Morocco', 'Mozambique', 'Mozambique/Malawi', 'Namibia', 'Niger', 'Nigeria', 'North Africa', 
        'Republic of Congo', 'Rwanda', 'Sahelian Countries', 'Sao Tome and Principe', 'Senegal', 
        'Seychelles', 'Sierra Leone', 'Sierra Leone/Guinea', 'Somalia', 'South Africa', 'South Sudan', 
        'Southern Africa', 'Sudan', 'Sudan/South Sudan', 'Swaziland', 'Tanzania', 'Togo', 'Tunisia', 
        'Uganda', 'West Africa', 'West/Central Africa', 'Zambia', 'Zimbabwe'
    ),
    'Asia': (
        'Afghanistan', 'Afghanistan/Pakistan', 'Afghanistan/Tajikistan', 'Armenia', 'Asia', 'Azerbaijan', 
        'Bangladesh', 'Bangladesh and India', 'Bangladesh/Myanmar', 'Bay of Bengal (India & Bangladesh)', 
        'Bhutan', 'Cambodia', 'China', 'DPR Korea', 'East Timor', 'East and South Asia', 'Georgia', 
        'India', 'India-Manipur', 'India/Bangladesh', 'India/Nepal', 'Indonesia', 'Iran', 'Iran/Iraq', 
        'Iran/Pakistan', 'Iraq', 'Israel', 'Jammu and Kashmir', 'Japan', 'Jordan', 
        'Jordan and neighboring countries', 'Kazakhstan', 'Kyrgyzstan', 'Kyrgyzstan/Uzbekistan', 
        'Lao PDR', 'Lao PDR/Cambodia', "Lao People's Democratic Republic", 'Lebanon', 'Malaysia', 
        'Maldives', 'Middle East', 'Mongolia', 'Myanmar', 'Nepal', 'Oman/Iran', 'Pakistan', 'Philippines', 
        'Philippines/Malaysia', 'Rep. of Korea', 'Saudi Arabia', 'South Asia', 'Southeast Asia', 
        'Sri Lanka', 'Syria', 'Syria/Iraq', 'Taiwan (China)', 'Tajikistan', 'Thailand', 'Timor-Leste', 
        'Turkey', 'Turkmenistan', 'Türkiye', 'Türkiye/Syria', 'Uzbekistan', 'Viet Nam', 'Yemen', 
        'occupied Palestinian territory'
    ),
    'Europe': (
        'Albania', 'Balkans', 'Belarus', 'Belarus/Russian Fed./Ukraine and Moldova', 
        'Bosnia and Herzegovina', 'Bulgaria', 'Central Europe', 'Central and Eastern Europe', 
        'Central/Eastern Europe', 'Central/Southern Europe', 'Cyprus', 'Czech Republic', 'Europe', 
        'FYR Macedonia', 'FYR of Macedonia', 'Former Yugoslav Republic of Macedonia', 'Greece', 
        'Hungary', 'Iceland', 'Italy', 'Madeira', 'Moldova', 'Montenegro', 'North Macedonia', 'Poland', 
        'Portugal', 'Romania', 'Russian Federation', 'Serbia', 'Serbia and Montenegro', 
        'Serbia and Montenegro and Romania', 'Slovak Republic', 'Slovakia', 'Slovenia', 
        'South-Eastern Europe', 'Spain', 'The former Yugoslav Republic of Macedonia', 'Ukraine', 
        'United Kingdom', 'Yugoslavia'
    ),
    'North America': (
        'Bahamas', 'Belize', 'Caribbean', 'Central America', 'Central America and the Caribbean', 
        'Costa Rica', 'Costa Rica/Panama', 'Cuba', 'Dominican Republic', 'Dominican Republic and Haiti', 
        'Dominican Republic/Haiti', 'Eastern Caribbean', 'El Salvador', 'El Salvador and Honduras', 
        'Grenada', 'Guatemala', 'Guatemala/Mexico', 'Haiti', 'Honduras', 'Honduras/Guatemala', 'Jamaica', 
        'Mexico', 'Mexico/Guatemala', 'Montserrat', 'Nicaragua', 'Panama', 'Saint Lucia', 
        'Saint Vincent and the Grenadines', 'Southwestern Caribbean', 'St Vincent & the Grenadines', 
        'St. Vincent', 'Trinidad and Tobago', 'USA', 'Virgin Islands'
    ),
    'South America': (
        'Argentina', 'Argentina/Paraguay', 'Argentina/Uruguay', 'Bolivia', 'Brazil', 'Brazil/Paraguay', 
        'Chile', 'Colombia', 'Ecuador', 'Ecuador/Peru', 'Guyana', 'Latin America', 'Paraguay', 'Peru', 
        'South America', 'Suriname', 'Uruguay', 'Venezuela'
    ),
    'Oceania': (
        'Australia', 'Cook Islands', 'Fiji', 'Fiji/Vanuatu', 'French Polynesia', 'Kiribati', 
        'Marshall Islands', 'Marshall Islands/Kiribati', 'Micronesia', 'New Caledonia', 'New Zealand', 
        'Pacific', 'Pacific Islands', 'Papua New Guinea', 'Samoa', 'Solomon Islands', 'South Pacific', 
        'Tonga', 'Tuvalu', 'Vanuatu'
    ),
    'Europe/Africa': (
        'Europe/Northern Africa',
    ),
    'Americas': (
        'Central and Latin America',
    )
}


In [35]:
# Flatten the grouped data into the original O(1) lookup dictionary format
continent_dict = {country: continent for continent, countries in continent_groups.items() for country in countries}

Map the dictionary to create our new column Continent


In [36]:
df['Continent'] = df['Country'].map(continent_dict)

# 3. finlly we Check if any rows failed to map
unmapped = df[df['Continent'].isna()]['Country'].unique()
if len(unmapped) > 0:
    print(f"Warning: These values were missing from the dictionary: {unmapped}")

  Group the disaster events by their major classification

In [37]:
disasters_by_category = {
    'Meteorological (Climate)': [
        'Cold Wave', 'Cold wave', 'Cyclone', 'Cyclone Ami', 'Cyclone Bingiza', 'Cyclone Boloeste', 
        'Cyclone Bondo', 'Cyclone Clovis', 'Cyclone Daman', 'Cyclone Eline and Gloria', 'Cyclone Elita', 
        'Cyclone Favio', 'Cyclone Gafilo', 'Cyclone Gonu', 'Cyclone Guba', 'Cyclone Heta', 
        'Cyclone Hubert', 'Cyclone Hudah', 'Cyclone Indlala', 'Cyclone Ivan', 'Cyclone Ivy', 
        'Cyclone Jokwe', 'Cyclone Laila', 'Cyclone Mala', 'Cyclone Mick', 'Cyclone Oli', 'Cyclone Rene', 
        'Cyclone Ului', 'Cyclonic Storm', 'Extratropical Cyclone', 'Florida Storm and Tornado', 
        'Hail Storm', 'Hail Storms', 'Hailstorm', 'Hailstorms', 'Heat Wave', 'Heavy Hail Storm', 
        'Heavy Rains', 'Heavy Snowfall and Cold Wave', 'Heavy Snowfalls', 'Heavy Storm', 'Heavy rains', 
        'Hurricane Beta', 'Hurricane Carlotta', 'Hurricane Charley', 'Hurricane Debby', 'Hurricane Dennis', 
        'Hurricane Emily', 'Hurricane Florence', 'Hurricane Floyd', 'Hurricane Georges', 'Hurricane Ignacio', 
        'Hurricane Irene', 'Hurricane Iris', 'Hurricane John', 'Hurricane Juliette', 'Hurricane Karl', 
        'Hurricane Katrina', 'Hurricane Keith and Tropical Storm Joyce', 'Hurricane Kenna', 'Hurricane Marty', 
        'Hurricane Michelle', 'Hurricane Norbert', 'Hurricane Olaf', 'Hurricane Stan', 'Hurricane Wilma', 
        'Local Storm', 'Rain and Windstorms', 'Rains', 'Rains and Snowfall', 'Rainy Season', 'Sandstorm', 
        'Severe Cold and Snowstorms', 'Severe Local Storm', 'Severe Local Storms', 'Severe Thunderstorms', 
        'Severe Weather', 'Severe Winter', 'Severe Winter Conditions', 'Severe storms', 'Snowfalls', 
        'Snowstorm', 'Snowstorms', 'Squall in Upper River Division', 'Storm', 'Storm Damage', 'Storm Surge', 
        'Storm in Upper River Division', 'Storm/Rain/Frost', 'Storms', 'Storms and Tornadoes', 
        'Strong Winds and Heavy Rains', 'Tornado', 'Torrential Rains', 'Tropical Cyclone', 
        'Tropical Cyclone Bijli', 'Tropical Cyclone Cora', 'Tropical Cyclone Dani', 'Tropical Cyclone Dina', 
        'Tropical Cyclone Fanele', 'Tropical Cyclone Gene', 'Tropical Cyclone Hamish', 'Tropical Cyclone Hary', 
        'Tropical Cyclone Jade', 'Tropical Cyclone Kesiny', 'Tropical Cyclone Lowell', 'Tropical Cyclone Manou', 
        'Tropical Cyclone Nargis', 'Tropical Cyclone Olaf', 'Tropical Cyclone Pat', 'Tropical Cyclone Sose', 
        'Tropical Cyclone Trina', 'Tropical Cyclone Vania', 'Tropical Cyclone Waka', 'Tropical Cyclones', 
        'Tropical Cyclones Yali and Zuman', 'Tropical Cyclonic Storm', 'Tropical Depression', 'Tropical Storm', 
        'Tropical Storm Adrian', 'Tropical Storm Aere', 'Tropical Storm Agatha', 'Tropical Storm Alma', 
        'Tropical Storm Alpha/Floods', 'Tropical Storm Andres', 'Tropical Storm Arlene', 'Tropical Storm Arthur', 
        'Tropical Storm Bilis', 'Tropical Storm Chantal', 'Tropical Storm Claudette', 'Tropical Storm Cyprien', 
        'Tropical Storm Dianmu', 'Tropical Storm Eric', 'Tropical Storm Ernesto', 'Tropical Storm Gamma', 
        'Tropical Storm Isidore', 'Tropical Storm Jeanne', 'Tropical Storm Jerry', 'Tropical Storm Lane', 
        'Tropical Storm Larry', 'Tropical Storm Lili', 'Tropical Storm Lorena', 'Tropical Storm Noul', 
        'Tropical Storm Odette', 'Tropical Storm Olga', 'Tropical Storm Otto', 'Tropical Storm Rashmi', 
        'Tropical Storm in Yumbi', 'Tropical Storms', 'Tropical Storms Ingrid and Manuel', 
        'Typhoon Aere and Typhoon Chaba', 'Typhoon Chebi', 'Typhoon Durian', 'Typhoon Lupit', 
        'Typhoon Mitag', 'Typhoon Neoguri', 'Typhoon Prapiroon', 'Typhoon Rusa', 'Typhoon Songda', 
        'Typhoon Sudal', 'Typhoon Talim', 'Typhoon Tokage', 'Typhoons and Tropical Storm', 'Violent Winds', 
        'Wind Storms', 'Wind and Hail Storm', 'Windstorm', 'Windstorm Surge', 'Windstorms', 
        'Xinjiang Snowfall', 'Zhejiang Typhoon Rananim'
    ],
    'Hydrological (Climate)': [
        'Avalanche', 'Avalanches', 'Avalanches and Floods', 'Avalanches and Heavy Snowfalls', 
        'Avalanches, Floods and Landslides', 'Coastal Flooding', 'Flash Flood', 'Flash Floods', 
        'Flash Floods and Hailstorms', 'Flash Floods and Landslide', 'Flash Floods and Landslides', 
        'Flash Floods and Mudslides', 'Flash Floods/Floods', 'Floods', 'Floods and Avalanche', 
        'Floods and Avalanches', 'Floods and Cold Wave', 'Floods and Drought', 'Floods and Flash Floods', 
        'Floods and Hailstorms', 'Floods and Heavy Snowfalls', 'Floods and High Tides', 
        'Floods and Landslide', 'Floods and Landslides', 'Floods and Landslides in Papua Province', 
        'Floods and Landslides in the Southwest', 'Floods and Mudflows', 'Floods and Mudslides', 
        'Floods and Violent Winds', 'Floods and Windstorm', 'Floods and landslides', 
        'Floods in Gansu Province', 'Floods in Mariental', 'Floods, Landslides and Strong Winds', 
        'Floods/Cyclone', 'Heavy Rains and Floods', 'High Tides and Floods', 'King Tides', 'Landslide', 
        'Landslide in Baglung', 'Landslides', 'Landslides and Floods', 'Mudslide', 'Mudslides', 
        'Mudslides and Floods', 'Rainstorms and Floods', 'River Floods', 'Severe Local Storms and Floods', 
        'Severe Sea Swell Floods', 'Storm and Floods', 'Storms and Floods', 'Tabasco and Chiapas Floods', 
        'Tindouf Floods', 'Yakutia Floods'
    ],
    'Climatological (Climate)': [
        '"Multiple Dzud" and Drought', 'Drought', 'Drought and Frost', 'Drought and Wild Fires', 'Dzud', 
        'Energy/Water/Food Insecurity', 'Environmental Emergency/Forest Fires', 'Fires and Haze', 
        'Food Insecurity', 'Forest Fire', 'Forest Fires', 'Water Crisis', 'Wild Fire', 'Wild Fires', 
        'Wild Fires/Heatwave', 'Wildfire', 'Wildfires'
    ],
    'Geophysical (Non-Climate)': [
        'Aceh Earthquake', 'Alor Earthquake', 'Ambae Volcano', 'Ambrym Volcano', 'Bingol Earthquake', 
        'Chaparrastique Volcano', 'Earthquake', 'Earthquake and Floods', 'Earthquake and Mudslides', 
        'Earthquake and Tsunami', 'Earthquake in Gansu Province', 'Earthquake in Inner Mongolia', 
        'Earthquake in Ludian county (Yunnan)', 'Earthquake in North Sulawesi', 
        'Earthquake in Qinghai Province', 'Earthquake in Sichuan Province', 'Earthquake in Tibet', 
        'Earthquake in Xinjiang Region', 'Earthquake in Xinjiang Uygur Autonomous Region', 
        'Earthquake in Yunnan Province', 'Earthquake in Yunnan and Sichuan Provinces', 
        'Earthquake on Sakhalin Island', 'Earthquake/Tsunami', 'Earthquakes', 
        'Earthquakes in Yunnan and Guizhou Provinces', 'Eyjafjallajökull Volcano', 'Fogo Volcano', 
        'Fuego Volcano', 'Galeras Volcano', 'Gansu Earthquake', 'Gaua Volcano', 'Hokkaido Earthquake', 
        'Ili Lewotolok Volcano', 'Kadovar Volcano', 'Karthala Volcanic Eruption', 'Karthala Volcano', 
        'La Soufrière Volcano', 'Langila Volcano', 'Lopevi Volcanic Eruption', 'Lopevi Volcano', 
        'Machín and Nevado del Huila Volcanoes', 'Manam Volcano', 'Manam Volcano Eruption', 'Mayon Volcano', 
        'Mentawai Earthquake', 'Monaro Volcano', 'Mount Merapi Volcano', 'Mt Agung Volcano', 
        'Mt Karai Volcano', 'Mt. Bromo Volcano', 'Mt. Bulusan Volcano', 'Mt. Gamalama Volcano', 
        'Mt. Kelud Volcano', 'Mt. Manam Volcano', 'Mt. Merapi Volcano', 'Mt. Sinabung Volcano', 
        'Mt. Ulawun Volcano', 'Nevado del Huila Volcano', 'Nevado del Huila Volcano Avalanches', 
        'North and West Sumatra Earthquake', 'Northern Molucca Sea Earthquake', 'Osh Earthquake', 
        'Pacaya Volcano', 'Papua Earthquake', 'Popocatépetl volcanic activity', 'Qeshm Earthquake', 
        'Sangay Volcano', 'Saravan Earthquake', 'Seismic swarm', 'Semeru Volcano', 'Sulawesi Earthquake', 
        'Sumatra Earthquakes and Tsunami', 'Sumbawa Earthquake', 'Taal Volcano', 'Tsunami', 
        'Tsunami/Earthquakes', 'Tungurahua Volcano', 'Ubinas Volcano', 'Volcanic Activity', 
        'Volcanic Activity Mt. Awu', 'Volcanic Activity Nyamulagira', 'Volcanic Eruption', 
        'Volcanic Eruption Mt. Egon', 'Volcanic Eruption Mt. Gamkonora', 'Volcanic Eruption and Tsunami', 
        'Volcano', 'Volcano Chaitén', 'Volcano Karangetang', 'Volcano Mt. Egon', 'Volcano Nyiragongo', 
        'Volcano Reventador', 'Volcano Tungurahua', 'Volcano de Colima', 'Volcano de Fuego', 
        'Volcanoes Guagua Pichincha and Tungurahua', 'West Java Earthquake', 
        'West Java Earthquake and Tsunami', 'West Sumatra Earthquakes', 'Yasur Volcano'
    ],
    'Biological (Non-Climate)': [
        'Acute Watery Diarrhoea (AWD) Outbreak', 'Anthrax Outbreak', 'Arboviral Outbreak', 
        'Armyworm Infestation', 'Chikungunya Outbreak', 'Chikungunya Virus Outbreak', 
        'Chikungunya and Dengue Outbreak', 'Chikungunya and Dengue Outbreaks', 
        'Chikungunya and Measles Outbreak', 'Cholera Epidemic', 'Cholera Outbreak', 
        'Cholera/Dysentery/Influenza Outbreaks', 'Cholera/Measles/Meningitis Outbreak', 
        'Dengue Fever Outbreak', 'Dengue Outbreak', 'Dengue and Yellow Fever Outbreaks', 
        'Diphtheria Outbreak', 'Ebola Outbreak', 'Hand/Foot/Mouth Disease Outbreak', 'Hepatitis E Outbreak', 
        'Insect Infestation', 'Lassa Fever Outbreak', 'Leishmaniasis Outbreak', 'Locust Infestation', 
        'Locusts', 'Malaria Outbreak', 'Malaria and Diphtheria Outbreaks', 'Marburg Disease Outbreak', 
        'Marburg Fever Outbreak', 'Marburg Virus Disease Outbreak', 'Measles', 'Measles Outbreak', 
        'Measles and Cholera Outbreaks', 'Measles and Rubella Outbreak', 'Meningitis Outbreak', 
        'Meningitis and Measles Outbreaks', 'Monkey Pox Outbreak', 'Mpox Outbreak', 'Plague Outbreak', 
        'Pneumonic Plague Outbreak', 'Polio Outbreak', 'Rift Valley Fever Outbreak', 'Yellow Fever Outbreak', 
        'Yellow Fever and Chikungunya Outbreaks'
    ],
    'Technological (Non-Climate)': [
        'Airplane Crash', 'Ammunition Dump Explosion', 'Bata Explosions', 'Beirut Port Explosions', 
        'Chemical Spill', 'Chemical Train Derailment and Explosion', 'Collapse of Dam/Floods', 
        'Cox’s Bazar Camp Settlement Fire', 'Dam Burst', 'Explosion', 'Explosion in Apeate', 
        'Explosion in Ryongchon County', 'Explosions', 'Ferry Disaster/Toxic Cargo', 'Fire', 'Fires', 
        'Fuel Truck Explosion', 'Galapagos Islands Oil Spill', 'Gas Well Explosion in Chongqing', 
        'Gas explosion/Fire', 'Hebei Spirit Oil Spill', 'MV Wakashio Oil Spill', 'Madrid Explosion', 
        'Mining Waste Spill', 'Oil Platform Accident', 'Oil Spill', 'Oil Tanker Explosion', 
        'River Pollution', 'School Collapse', 'Sidoarjo mudflow and gas pipe explosion', 
        'Technical Accident and Floods', 'Toxic Pollution', 'Toxic Spill', 'Urban Fire Incident', 
        'Waste Spill', 'Waste Water Treatment Plant Floods'
    ]
}



In [38]:
#  Automatically generate the flat dictionary using a comprehension
disaster_dict = {
    event: category 
    for category, events in disasters_by_category.items() 
    for event in events
}

In [39]:
#  Map the dictionary to create a standard parent category column
df['Category'] = df['Disaster_Type'].map(disaster_dict)

# 3. Create a simple "Climate vs Non-Climate" binary column for the visualizations we discussed earlier
df['Disaster_status'] = df['Category'].apply(lambda x: 'Climate-Driven' if str(x).endswith('(Climate)') else 'Non-Climate')

print(df[['Disaster_Type', 'Category', 'Disaster_status']].head())

  Disaster_Type                   Category Disaster_status
0    Earthquake  Geophysical (Non-Climate)     Non-Climate
1    Earthquake  Geophysical (Non-Climate)     Non-Climate
2        Floods     Hydrological (Climate)  Climate-Driven
3  Flash Floods     Hydrological (Climate)  Climate-Driven
4  Flash Floods     Hydrological (Climate)  Climate-Driven


In [40]:
df.head(3)
df.info()

<class 'pandas.DataFrame'>
Index: 11590 entries, 0 to 11798
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Country          11590 non-null  str           
 1   Disaster_Type    11590 non-null  str           
 2   Date             11590 non-null  datetime64[us]
 3   Original_Title   11590 non-null  str           
 4   Continent        11590 non-null  str           
 5   Category         11589 non-null  str           
 6   Disaster_status  11590 non-null  str           
dtypes: datetime64[us](1), str(6)
memory usage: 724.4 KB


the insight we will be deriving from this dataset is how often are climate 
related disasters happening compared to other accidents and the frequecy 
of this climate related disasters and make a conclusioon about climate change by 
country, continent and gloabally

convert our datetime datatype  column to a year standard

In [41]:
# df['Year'] = pd.to_datetime(df['Date']).dt.year
df['Year'] = pd.to_datetime(df['Date'], errors='coerce').dt.year
df.info()
# df['Year'] = df['Year'].apply(pd.to_, format='mixed')
# df.dtypes
df.head(2)

<class 'pandas.DataFrame'>
Index: 11590 entries, 0 to 11798
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Country          11590 non-null  str           
 1   Disaster_Type    11590 non-null  str           
 2   Date             11590 non-null  datetime64[us]
 3   Original_Title   11590 non-null  str           
 4   Continent        11590 non-null  str           
 5   Category         11589 non-null  str           
 6   Disaster_status  11590 non-null  str           
 7   Year             11590 non-null  int32         
dtypes: datetime64[us](1), int32(1), str(6)
memory usage: 769.6 KB


,Country,Disaster_Type,Date,Original_Title,Continent,Category,Disaster_status,Year
0,Indonesia,Earthquake,2026-08-01,Indonesia: Earthquake - Aug 2026,Asia,Geophysical (Non-Climate),Non-Climate,2026
1,Colombia,Earthquake,2026-08-01,Colombia: Earthquake - Aug 2026,South America,Geophysical (Non-Climate),Non-Climate,2026


finally we select 'Country', 'Disaster_Type', 'Continent', 'Year', 'Category', 'Disaster_status'
columns to be saved for our data analysis phase

In [42]:
df = pd.DataFrame(df, columns=['Country', 'Disaster_Type', 'Continent', 'Year', 'Category', 'Disaster_status'])

df.head(10)

,Country,Disaster_Type,Continent,Year,Category,Disaster_status
0,Indonesia,Earthquake,Asia,2026,Geophysical (Non-Climate),Non-Climate
1,Colombia,Earthquake,South America,2026,Geophysical (Non-Climate),Non-Climate
2,Myanmar,Floods,Asia,2026,Hydrological (Climate),Climate-Driven
3,Afghanistan,Flash Floods,Asia,2026,Hydrological (Climate),Climate-Driven
4,India,Flash Floods,Asia,2026,Hydrological (Climate),Climate-Driven
5,Kyrgyzstan,Flash Floods,Asia,2026,Hydrological (Climate),Climate-Driven
6,Tajikistan,Floods,Asia,2026,Hydrological (Climate),Climate-Driven
7,Bangladesh,Flash Floods,Asia,2026,Hydrological (Climate),Climate-Driven
8,Georgia,Floods,Asia,2026,Hydrological (Climate),Climate-Driven
9,Central African Republic,Cholera Outbreak,Africa,2026,Biological (Non-Climate),Non-Climate


In [43]:
df.to_csv(r'/home/ab303el/Documents/virtual internship/CodeAlpha Internship/data/clean_data.csv', index=False)
print('data saved successfully')

data saved successfully
